In [0]:
dbutils.library.restartPython()

In [0]:

import json
from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.generic_functions import (
    rename_columns,
    casting,
    drop_columns
)
from common_utils.scd_type2 import scd_type2

# =========================================
# 0. Logger
# =========================================

logger = get_logger("b2s_FactWork.ipynb")
logger.info("Session started")

# =========================================
# 1. CONFIG PATH
# =========================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Setting configurations")

# =========================================
# 2. Read Config
# =========================================

with open(config_path, "r") as file:
    config = json.load(file)

# =========================================
# 3. CONFIG Sections
# =========================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]
source_table = source_config["source_table"]
target_table = target_config["target_table"]

logger.info("Configuration completed")

# SCD type is nested under "scd" in the config and stored as a string
scd_type = int(config.get("scd", {}).get("type", 1))

logger.info("Configuration completed. SCD type = %s", scd_type)


# =========================================
# 4. Fact work Transformations
# =========================================

def factwork_transformations(df, config):

    # -------------------------------------
    # Trim and select columns
    # -------------------------------------

    df = df.select(
        F.col("work_id"),
        F.col("people_id"),
        F.col("county_id"),
        F.trim(F.col("status")).alias("status"),
        F.col("cancellation_id"),
        F.col("scheduled_start"),
        F.col("scheduled_end"),
        F.col("source_file"),
        F.col("current_timestamp")
    )

    
    # -------------------------------------
    # Cast columns
    # -------------------------------------

    df = casting(
        df,
        config.get("cast_columns", {})
    )

    # -------------------------------------
    # Rename columns
    # -------------------------------------

    df = rename_columns(
        df,
        config.get("rename_column", {})
    )


    # -------------------------------------
    # Drop columns
    # -------------------------------------

    df = drop_columns(
        df,
        config.get("drop_columns", [])
    )
    # -------------------------------------
    # Droping duplicates of the key columns
    # -------------------------------------

    df = df.dropDuplicates(config["keys"])
                           

    # -------------------------------------
    # Select final columns
    # -------------------------------------

    logger.info(
        "[%s] Selecting final Silver columns",
        source_name
    )

    df = df.select(*config["select_columns"])

    return df


# =========================================
# 5. Reading Bronze Table
# =========================================

logger.info("Reading from source table [%s]", source_table)

df = spark.read.table(source_table)


# =========================================
# 6. Run Transformations
# =========================================

cleaned_df = factwork_transformations(
    df,
    config
)

logger.info(
    "[%s] County transformation completed",
    source_name
)


# =========================================
# 7. Data Quality Checks
# =========================================

logger.info("[%s] Running data quality checks", source_name)

# Check null primary keys and county names
# Get composite keys from config

key_columns = config.get("keys",[])

if len(key_columns) != 2:
    raise ValueError("Expected two composite key columns in config")



# -------------------------------------
# Check null primary keys
# -------------------------------------

invalid_records = cleaned_df.filter(
    F.col(key_columns[0]).isNull()
    | F.col(key_columns[1]).isNull()
)

# -------------------------------------
# Check duplicate composite keys
# -------------------------------------


duplicates = (
    cleaned_df
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

# -------------------------------------
# Validate records
# -------------------------------------



if invalid_records.limit(1).count() > 0:
    raise ValueError("Null or invalid Fact work records found")

if duplicates.limit(1).count() > 0:
    raise ValueError("Duplicate records found")

logger.info("Fact_Work data quality checks passed")



# ============================================================
# 8. Load into Silver (SCD Type 1 / SCD Type 2)
# ============================================================

if scd_type not in (1, 2):
    raise ValueError(
        f"Unsupported SCD type [{scd_type}] "
        f"for source [{source_name}]"
    )

# Create schema from target_table
target_schema = ".".join(target_table.split(".")[:-1])
if target_schema:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

if scd_type == 2:
    if not spark.catalog.tableExists(target_table):
        # Initial load performed here: the helper's initial-load write
        # mode is not supported on this compute. This mirrors scd_type2's
        # initial load (same columns, same values).
        logger.info(
            "[%s] Target not found. Initial load (SCD Type 2)",
            source_name
        )

        run_ts = spark.sql(
            "SELECT current_timestamp() AS run_ts"
        ).first()["run_ts"]

        initial_load = (
            cleaned_df
            .withColumn("valid_from", F.lit(run_ts).cast("timestamp"))
            .withColumn(
                "valid_to",
                F.to_timestamp(F.lit("9999-12-31 23:59:59"))
            )
            .withColumn("is_current", F.lit(True))
        )

        (
            initial_load
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        logger.info("[%s] Starting SCD Type 2", source_name)

        scd_type2(
            spark,
            cleaned_df,
            target_table,
            key_columns
        )

else:
    if spark.catalog.tableExists(target_table):
        logger.info("[%s] Starting SCD Type 1 (upsert)", source_name)

        merge_condition = " AND ".join(
            f"t.`{k}` = s.`{k}`" for k in key_columns
        )

        (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(cleaned_df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        logger.info("[%s] Target not found. Initial load", source_name)

        (
            cleaned_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

logger.info(
    "[%s] Successfully loaded into Silver: %s",
    source_name,
    target_table
)